# Проверка блоков 0 и 1

Ноутбук вызывает функции из `cards.py` и `evaluator.py`.

- Блок 0: `Card`, `make_deck`, `parse_cards`, `cards_to_str`, `remove_known`, `group_by_rank`, `group_by_suit`.
- Блок 1: `evaluate_5`, `evaluate` (для 5, 6 и 7 карт), `hand_category`, `describe_hand`, `compare`.

`parse_cards` ждёт карты через пробел: `"As Kh Qd"`.


In [1]:
from cards import (
    Card,
    cards_to_str,
    group_by_rank,
    group_by_suit,
    make_deck,
    parse_cards,
    remove_known,
)
from combo import unpack
from evaluator import (
    HandCategory,
    compare,
    describe_hand,
    evaluate,
    evaluate_5,
    hand_category,
)

import random


In [2]:
ace_spades = Card(12, 0)
assert ace_spades == Card("A", "s") == Card("a", "S")
assert str(ace_spades) == "As"
assert repr(ace_spades) == "Card(As)"
assert int(ace_spades) == 48
assert ace_spades == 48
assert ace_spades != Card("A", "h")
assert ace_spades != "As"

assert Card(0, 0) < Card(0, 1) < Card(1, 0)
assert Card("2", "s") <= Card("2", "h") <= Card("2", "h")
assert Card("K", "c") > Card("Q", "s")
assert hash(Card("A", "s")) == hash(Card(12, 0))
assert len({Card("A", "s"), Card("a", "S"), Card("K", "h")}) == 2

for bad in [(13, 0), (-1, 0), (0, 4), ("1", "s"), ("A", "x"), ("As", "s")]:
    try:
        Card(*bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"ожидался ValueError для Card{bad}")

print("Card: ok")


Card: ok


In [3]:
deck = make_deck()
assert len(deck) == 52
assert len(set(deck)) == 52
assert cards_to_str(deck[:3]) == "2s 2h 2d"
assert cards_to_str(deck[-1:]) == "Ac"

hand = parse_cards("As Kh Qd")
assert hand == [Card("A", "s"), Card("K", "h"), Card("Q", "d")]
assert cards_to_str(hand) == "As Kh Qd"

known = parse_cards("As Kh")
rest = remove_known(deck, known)
assert len(deck) == 52
assert len(rest) == 50
assert all(card not in rest for card in known)
assert Card("Q", "d") in rest

pairs = group_by_rank(parse_cards("As Ah Kd Kc 2s"))
assert {len(cards) for cards in pairs.values()} == {2, 1}
assert len(pairs[Card("A", "s").rank]) == 2
assert len(pairs[Card("2", "c").rank]) == 1

suits = group_by_suit(parse_cards("As Ks Qs Jd Td"))
assert len(suits[Card("A", "s").suit]) == 3
assert len(suits[Card("J", "d").suit]) == 2
assert Card("K", "h").suit not in suits

print("helpers: ok")


helpers: ok


In [4]:
def hand(*names: str) -> list[Card]:
    cards = parse_cards(" ".join(names))
    random.shuffle(cards)
    return cards


def packed(category: int, *kickers: int) -> tuple[int, ...]:
    return (category, *kickers, *([0] * (5 - len(kickers))))


def got(cards: list[Card]) -> tuple[int, ...]:
    return unpack(evaluate_5(cards))


# 9 — роял-флеш, 8 — стрит-флеш (wheel = пятёрка, ранг 3)
assert got(hand("Ts", "Js", "Qs", "Ks", "As")) == packed(9)
assert got(hand("As", "2s", "3s", "4s", "5s")) == packed(8, 3)
assert got(hand("5c", "6c", "7c", "8c", "9c")) == packed(8, 7)

# 7 — каре, 6 — фулл-хаус
assert got(hand("Ks", "Kh", "Kd", "Kc", "2s")) == packed(7, 11, 0)
assert got(hand("Ks", "Kh", "Kd", "Kc", "As")) == packed(7, 11, 12)
assert got(hand("8s", "8h", "8d", "4c", "4s")) == packed(6, 6, 2)
assert got(hand("4s", "4h", "4d", "8c", "8s")) == packed(6, 2, 6)

# 5 — флеш, 4 — стрит (wheel не старше шестёрки)
assert got(hand("2s", "5s", "9s", "Js", "As")) == packed(5, 12, 9, 7, 3, 0)
assert got(hand("Ac", "2d", "3s", "4h", "5c")) == packed(4, 3)
assert got(hand("Tc", "Jd", "Qs", "Kh", "Ac")) == packed(4, 12)
assert got(hand("5c", "6d", "7s", "8h", "9c")) == packed(4, 7)

# 3 — сет, 2 — две пары, 1 — пара, 0 — старшая карта
assert got(hand("Qs", "Qh", "Qd", "2c", "7s")) == packed(3, 10, 5, 0)
assert got(hand("2s", "2h", "2d", "3c", "7s")) == packed(3, 0, 5, 1)
assert got(hand("Js", "Jh", "8d", "8c", "2s")) == packed(2, 9, 6, 0)
assert got(hand("3s", "3h", "2d", "2c", "As")) == packed(2, 1, 0, 12)
assert got(hand("As", "Ah", "Kd", "Qc", "Js")) == packed(1, 12, 11, 10, 9)
assert got(hand("5s", "5h", "7d", "8c", "9s")) == packed(1, 3, 7, 6, 5)
assert got(hand("As", "Kh", "Qd", "Jc", "9s")) == packed(0, 12, 11, 10, 9, 7)
assert got(hand("As", "2h", "3d", "4c", "6s")) == packed(0, 12, 4, 2, 1, 0)

wheel_sf = evaluate_5(parse_cards("As 2s 3s 4s 5s"))
royal = evaluate_5(parse_cards("Ts Js Qs Ks As"))
assert compare(wheel_sf, royal) == -1
assert compare(royal, wheel_sf) == 1
assert compare(royal, evaluate_5(parse_cards("As Ks Qs Js Ts"))) == 0

assert compare(
    evaluate_5(parse_cards("8s 8h 8d 4c 4s")),
    evaluate_5(parse_cards("2s 5s 9s Js As")),
) == 1
assert compare(
    evaluate_5(parse_cards("2s 2h 2d 2c As")),
    evaluate_5(parse_cards("2s 2h 2d 2c 3s")),
) == 1
assert compare(
    evaluate_5(parse_cards("As Ah Kd Kc 2s")),
    evaluate_5(parse_cards("Qs Qh Kd Kc 2s")),
) == 1
assert compare(
    evaluate_5(parse_cards("Ts 8c 6d 4h 2c")),
    evaluate_5(parse_cards("9s 8c 6d 4h 2c")),
) == 1

# 6 и 7 карт: лучшая пятёрка, а не порядок в списке
seven = parse_cards("2c 2d 2h 2s 3c As Kd")
assert evaluate(seven) == evaluate_5(parse_cards("2c 2d 2h 2s As"))
assert hand_category(evaluate(seven)) == HandCategory.FOUR_OF_A_KIND

full = parse_cards("Ah Ad Ac Kd Kc Qs 2c")
assert unpack(evaluate(full)) == packed(6, 12, 11)
assert hand_category(evaluate(full)) == HandCategory.FULL_HOUSE

straight = parse_cards("2c 3d 4h 5s 9c 6d Kh")
assert evaluate(straight) == evaluate_5(parse_cards("2c 3d 4h 5s 6d"))

flush = parse_cards("As 2s 5s 9s Js Kd")
assert evaluate(flush) == evaluate_5(parse_cards("As 2s 5s 9s Js"))
assert hand_category(evaluate(parse_cards("Ts Js Qs Ks As 2h"))) == HandCategory.ROYAL_FLUSH

five = parse_cards("As Kh Qd Jc 9s")
assert evaluate(five) == evaluate_5(five)

fh = evaluate_5(parse_cards("8s 8h 8d 4c 4s"))
assert describe_hand(fh) == "6: 6, 2, 0, 0, 0"

for bad in (parse_cards("As Kh"), parse_cards("As Kh Qd Jc")):
    try:
        evaluate_5(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("evaluate_5 должен требовать 5 карт")

try:
    evaluate(parse_cards("As Kh Qd Jc"))
except ValueError:
    pass
else:
    raise AssertionError("evaluate должен требовать хотя бы 5 карт")

print("evaluator: ok")


evaluator: ok


# 2 deck

In [5]:
from deck import Deck

In [6]:
deck = Deck(shuffle=False)
assert len(deck) == 52
assert len(set(deck)) == 52
assert cards_to_str(deck[:3]) == "2s 2h 2d"
assert cards_to_str(deck[-1:]) == "Ac"
assert cards_to_str(deck.draw(2)) == "2s 2h"
assert cards_to_str(deck[:3]) == "2d 2c 3s"

deck.remove(parse_cards("As Kh"))
assert len(deck) == 48

deck.remake()
assert len(deck) == 52
deck_before = deck[:]
deck.remake()
assert deck_before != deck

try:
    deck.draw(len(deck) + 1)
except ValueError:
    pass
else:
    raise AssertionError("draw больше колоды должен давать ValueError")


print("deck: ok")

deck: ok
